# `integrate_results.csv`

Connor Shrader / 09-21-2025

This file loads in and computes metrics on all of the clone results and saves to the file `all_clone_results.csv`.

In [1]:
import os
import pandas as pd
import numpy as np
import ast
import testis_model as tm
from tqdm import tqdm

from saving_loading import find_scenarios, load_all_scenarios, load_clone_names
from testis_model.metrics import init_time, last_time, min_time, max_time, freq_at, size, avg_freq_change, run_likelihood, n_transient

In [2]:
scenarios = find_scenarios()
fish_data, clone_data = load_all_scenarios(scenarios)
# clone_names = load_clone_names()

In [3]:
print(f'Found scenarios are: {scenarios}.')

Found scenarios are: ['r36_rp5_mimic', 'r36_rp15', 'r36_rp1.5', 'r36_rp12', 'r36_rp3.5_mimic', 'r18_rp3.5_mimic', 'r36_rp7_mimic', 'r36_rp13', 'r36_rp14', 'r36_rp25', 'r36_rp2.5_mimic', 'r36_rp6', 'r36_rp3_mimic', 'r36_rp1', 'r36_rp8', 'r18_rp2.5_mimic', 'r36_rp9', 'no_collapse', 'r36_rp0', 'r36_rp7', 'r36_rp25_mimic', 'r18_rp8_mimic', 'r36_rp1_mimic', 'r18_rp0.5_mimic', 'r36_rp0.5_mimic', 'r36_rp2.5', 'r36_rp6_mimic', 'r36_rp4_mimic', 'r36_rp20_mimic', 'full_pipeline_trunc', 'r36_rp36', 'r36_rp0_mimic', 'r18_rp9_mimic', 'r36_rp30', 'r18_rp1.5_mimic', 'r36_rp2_mimic', 'r36_rp1.5_mimic', 'r36_rp3.5', 'r36_rp20', 'r36_rp11', 'r18_rp4.5_mimic', 'r18_rp13_mimic', 'r18_rp7_mimic', 'r36_rp4.5_mimic', 'r18_rp5_mimic', 'r18_rp11_mimic', 'r36_rp10', 'r36_rp15_mimic', 'r36_rp2', 'r18_rp1_mimic', 'r18_rp15_mimic', 'r36_rp5', 'r36_rp11_mimic', 'r36_rp8_mimic', 'r36_rp13_mimic', 'r18_rp3_mimic', 'r36_rp4', 'r36_rp3', 'r36_rp14_mimic', 'r18_rp4_mimic', 'r18_rp10_mimic', 'full_pipeline', 'r36_rp0.5',

In [4]:
'r36_rp_5_mimic' 

'r36_rp_5_mimic'

In [5]:
long_dfs = []
for scenario in scenarios:
    path = os.path.join('scenarios', scenario, f'{scenario}_long.csv')
    if os.path.isfile(path):
        long_dfs.append(pd.read_csv(path))

for i, df in enumerate(long_dfs):
    if "duplicate" not in df.columns:
        df["duplicate"] = 0
        long_dfs[i] = df

In [6]:
long_results = pd.concat(long_dfs).set_index(['scenario', 'fish', 'duplicate', 'clone'])
print(f'Long results has length {len(long_results)}.')
long_results.head()

Long results has length 83993.


init_time  init_freq     r_opt  \
scenario      fish duplicate clone                                   
r36_rp5_mimic 1    0         0            5.0   0.809322  0.734669   
                             1            5.0   0.096751  4.112847   
                             2            5.0   0.062147  2.283607   
                             3            5.0   0.021186  2.399909   
                             4            5.0   0.010593  2.053634   

                                         l_max     lower      upper  \
scenario      fish duplicate clone                                    
r36_rp5_mimic 1    0         0     -137.014545  0.231772   2.631180   
                             1     -133.666750  1.773068  11.750108   
                             2     -126.116370  0.888735   6.989279   
                             3      -35.368118  0.704391   9.402862   
                             4      -38.379028  0.551991   6.869552   

                                        l_null  
scenario      fish duplicate clone              
r36_rp5_mimic 1    0         0     -156.420841  
                             1     -348.164613  
                             2     -289.429360  
                             3     -125.333891  
                             4      -78.825582

In [7]:
# Specify order of columns
columns = ['scenario', 'fish', 'duplicate', 'clone', 'init_time', 'init_freq', 'last_time',
           'min_time', 'max_time', 'max_freq', 'n_transient', 'size', 'avg_freq_change', 'r_opt',
           'l_max', 'lower', 'upper', 'l_null']

rows = []
for (scenario, fish, duplicate, clone), data in tqdm(clone_data.items()):
    try:
        if (scenario, fish, duplicate, clone) in long_results.index:
            row = long_results.loc[scenario, fish, duplicate, clone].to_dict()
        else:
            row = {col: pd.NA for col in columns}
            it = init_time(data)
            if it is None: continue
            else: row['init_time'] = it

            row['init_freq'] = freq_at(data, row['init_time'])
    
        row['scenario'] = scenario
        row['fish'] = fish
        row['duplicate'] = duplicate
        row['clone'] = clone
        
        row['last_time'] = last_time(data)
        row['min_time'] = min_time(data)
        row['max_time'] = max_time(data)
        row['max_freq'] = freq_at(data, row['max_time'])
        row['n_transient'] = n_transient(data)
        row['size'] = size(data)
            
        row['avg_freq_change'] = avg_freq_change(data)
        
        rows.append(row)

    except Exception as e:
        print(f'Error for scenario={scenario}, fish={fish}, clone={clone}')
        raise

100%|██████████████████████████████████| 121498/121498 [02:55<00:00, 694.14it/s]


In [8]:
df = pd.DataFrame(rows, columns = columns)
df.to_csv('all_clone_results.csv', index = False)

In [9]:
num_na = len(df[df['r_opt'].isna()])
num_fail = len(df[df['l_max'] < -1e5])

print(f'{num_na} rows have r_opt == NA, and {num_fail} have l_max < -1e5 (indicating a failure to converge).')

1659 rows have r_opt == NA, and 0 have l_max < -1e5 (indicating a failure to converge).
